In [ ]:
# Get presigned URL and download file
RESPONSE=$(curl -X POST "https://mozilladatacollective.com/api/datasets/cmu5wui2w00e8nq07idithpm6/download" \
  -H "Authorization: Bearer YOUR_API_KEY" \
  -H "Content-Type: application/json")

# Extract download URL and download file
DOWNLOAD_URL=$(echo $RESPONSE | jq -r '.downloadUrl')
curl -o "common-voice-scripted-speech-27-0-portug-c1c03212.tar.gz" "$DOWNLOAD_URL"

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"

# Identifica o nome do arquivo baixado
# e extrai apenas o arquivo 'validated.tsv' diretamente para a pasta de dados

import os
import glob

# Procura por qualquer arquivo .tar.gz no diretório atual
tar_files = glob.glob("*.tar.gz")

if tar_files:
    tar_file = tar_files[0]
    print(f"Arquivo encontrado: {tar_file}")
    print("Buscando a localização de 'validated.tsv' dentro do pacote...")
    
    # Lista os arquivos dentro do tar para achar o caminho exato do validated.tsv
    import tarfile
    with tarfile.open(tar_file, "r:gz") as tar:
        # Localiza o arquivo validated.tsv dentro do pacote
        validated_member = [m for m in tar.getmembers() if "validated.tsv" in m.name]
        if validated_member:
            member = validated_member[0]
            print(f"Extraindo '{member.name}'...")
            
            # Extrai apenas este arquivo específico
            tar.extract(member, path=".")
            
            origem = Path(member.name)
            destino = DATA_DIR / "validated.tsv"
            
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            
            origem.rename(destino)
            print(f"Pronto! O arquivo foi extraído e salvo em: {destino}")
        else:
            print("Não foi possível encontrar o arquivo 'validated.tsv' dentro do pacote compactado.")

In [ ]:
import pandas as pd

# Caminho do arquivo extraído
validated_path = DATA_DIR / "validated.tsv"

validated = pd.read_csv(validated_path, sep="\t")

print("Total de registros carregados:", validated.shape[0])
validated.head()

In [ ]:
validated1 = validated.dropna(subset=['accents'])

print("Total de registros com label:", validated1.shape[0])

In [ ]:
# Filtrando sentenças que contêm pelo menos 3 palavras
min_palavras = 3
validated_filtrado = validated1[validated1['sentence'].apply(lambda x: len(str(x).strip().split()) >= min_palavras)]

print(f"Total original de registros com label: {validated1.shape[0]}")
print(f"Total após filtrar por pelo menos {min_palavras} palavras: {validated_filtrado.shape[0]}")

validated_filtrado.head()

In [ ]:
validated_filtrado['accents'].dropna().unique()